# Recipe GPT-2 Training - Comprehensive Dataset\nTraining on 120,773 recipes from FoodNetwork, Epicurious, and AllRecipes\n\n**Upload File:** recipe_training_COMPREHENSIVE.txt (141.7 MB)

## Step 1: Check GPU

In [ ]:
import torch\nprint(f'GPU Available: {torch.cuda.is_available()}')\nif torch.cuda.is_available():\n    print(f'GPU Name: {torch.cuda.get_device_name(0)}')\n    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')\nelse:\n    print('⚠️ WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

## Step 2: Install Dependencies

In [ ]:
!pip install transformers datasets accelerate -q

## Step 3: Upload Training Data\n\nClick the folder icon (📁) on the left and upload:\n- `recipe_training_COMPREHENSIVE.txt` (~142 MB)\n\nWait for upload to complete before proceeding.

## Step 4: Verify Upload

In [ ]:
import os\nfilename = 'recipe_training_COMPREHENSIVE.txt'\nif os.path.exists(filename):\n    size = os.path.getsize(filename) / 1024 / 1024\n    print(f'✅ File uploaded: {size:.1f} MB')\n    with open(filename, 'r', encoding='utf-8') as f:\n        content = f.read()\n        recipes = content.count('<END>')\n        print(f'✅ Recipes found: {recipes:,}')\n        print(f'\\nSample (first 300 chars):\\n{content[:300]}...')\nelse:\n    print(f'❌ File not found! Please upload {filename}')

## Step 5: Initialize Model and Training

In [ ]:
from transformers import (\n    GPT2LMHeadModel,\n    GPT2Tokenizer,\n    TextDataset,\n    DataCollatorForLanguageModeling,\n    Trainer,\n    TrainingArguments\n)\n\nprint('='*60)\nprint('INITIALIZING TRAINING')\nprint('='*60)\n\n# Load tokenizer\nprint('\\nLoading tokenizer...')\ntokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')\ntokenizer.pad_token = tokenizer.eos_token\n\n# Load base model (starting from scratch with DistilGPT2)\nprint('Loading base model (DistilGPT2)...')\nmodel = GPT2LMHeadModel.from_pretrained('distilgpt2')\nprint(f'Model parameters: {model.num_parameters():,}')\n\n# Load dataset\nprint('\\nLoading training dataset...')\ntrain_dataset = TextDataset(\n    tokenizer=tokenizer,\n    file_path='recipe_training_COMPREHENSIVE.txt',\n    block_size=128\n)\nprint(f'Training samples: {len(train_dataset):,}')\n\n# Data collator\ndata_collator = DataCollatorForLanguageModeling(\n    tokenizer=tokenizer,\n    mlm=False\n)\n\n# Training configuration (optimized for GPU)\nprint('\\nSetting up training configuration...')\ntraining_args = TrainingArguments(\n    output_dir='recipe_gpt2_trained',\n    overwrite_output_dir=True,\n    num_train_epochs=2,\n    per_device_train_batch_size=8,  # GPU can handle larger batches\n    gradient_accumulation_steps=2,\n    save_steps=1000,\n    save_total_limit=2,\n    logging_steps=50,\n    fp16=True,  # Mixed precision for faster training\n    learning_rate=5e-5,\n    warmup_steps=500,\n    weight_decay=0.01\n)\n\n# Initialize trainer\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    data_collator=data_collator,\n    train_dataset=train_dataset\n)\n\nprint('\\n' + '='*60)\nprint('✅ READY TO TRAIN')\nprint('='*60)\nprint('Estimated time: 2-3 hours on GPU')\nprint('Progress will update every 50 steps')\nprint('='*60)

## Step 6: Start Training (This cell takes 2-3 hours)

In [ ]:
print('🔥 STARTING TRAINING...')\nprint('This will take 2-3 hours. Keep this tab open!')\nprint()\n\ntrainer.train()\n\nprint('\\n' + '='*60)\nprint('✅ TRAINING COMPLETE!')\nprint('='*60)

## Step 7: Save and Download Model

In [ ]:
print('💾 Saving final model...')\nmodel.save_pretrained('recipe_gpt2_final')\ntokenizer.save_pretrained('recipe_gpt2_final')\nprint('✅ Model saved!')\n\nprint('\\n📦 Creating ZIP file...')\n!zip -r recipe_gpt2_final.zip recipe_gpt2_final/\n\nprint('\\n' + '='*60)\nprint('✅ ALL DONE!')\nprint('='*60)\nprint('\\n📥 Download recipe_gpt2_final.zip from the files panel')\nprint('📁 Extract to: pythonML/app/models/recipe_gpt2/')\nprint('🔄 Restart your Python backend')\nprint('🎉 Test with improved recipe generation!')\nprint('='*60)

## Optional: Test the Model\n\nQuick test before downloading:

In [ ]:
# Quick test\ntest_input = 'INPUT: kimchi, tofu, rice\\nOUTPUT:'\ninputs = tokenizer(test_input, return_tensors='pt').to('cuda')\noutputs = model.generate(\n    inputs['input_ids'],\n    max_length=300,\n    temperature=0.7,\n    top_p=0.85,\n    do_sample=True\n)\nresult = tokenizer.decode(outputs[0], skip_special_tokens=True)\nprint('='*60)\nprint('TEST RECIPE GENERATION')\nprint('='*60)\nprint(result)\nprint('='*60)